# 01 — Data Cleaning Pipeline

This notebook runs the full data-cleaning pipeline from `src.data_cleaning`.
It validates row counts per table, inspects column dtypes, and surfaces any
data-quality issues (nulls, duplicate keys, unexpected values) present in the
raw Dunnhumby Complete Journey files before downstream modeling.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print('sys.path set — imports ready')

In [ ]:
# Load processed parquets to inspect cleaning results
from src.data_cleaning import PROCESSED_DIR

tables = [
    'products', 'transactions', 'demographics',
    'campaign_desc', 'campaign_table', 'coupon', 'coupon_redempt'
]

records = []
for t in tables:
    df = pd.read_parquet(PROCESSED_DIR / f'{t}.parquet')
    records.append({
        'table': t,
        'rows': len(df),
        'columns': len(df.columns),
        'null_cells': int(df.isnull().sum().sum()),
        'null_pct': round(df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100, 2)
    })

summary = pd.DataFrame(records)
print(summary.to_string(index=False))

In [ ]:
# Transaction-level quality checks
txn = pd.read_parquet(PROCESSED_DIR / 'transactions.parquet')
print(f'Transaction rows       : {len(txn):,}')
print(f'Unique households      : {txn["household_key"].nunique():,}')
print(f'Unique products        : {txn["product_id"].nunique():,}')
print(f'Date span (days)       : {txn["day"].min()} - {txn["day"].max()} ({txn["day"].max() - txn["day"].min() + 1} days)')
print(f'Week span              : {txn["week_no"].min()} - {txn["week_no"].max()}')
print(f'Negative sales_value   : {(txn["sales_value"] < 0).sum():,}')
print(f'Zero sales_value       : {(txn["sales_value"] == 0).sum():,}')
print(f'Total revenue          : ${txn["sales_value"].sum():,.0f}')

In [ ]:
# Demographics coverage
demo = pd.read_parquet(PROCESSED_DIR / 'demographics.parquet')
print(f'Households with demographics : {len(demo):,}')
print(f'Columns                      : {list(demo.columns)}')
print()
print('Age distribution:')
if 'age_desc' in demo.columns:
    print(demo['age_desc'].value_counts().to_string())

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pathlib

CHARTS = pathlib.Path('..') / 'outputs' / 'charts'
CHARTS.mkdir(parents=True, exist_ok=True)

txn = pd.read_parquet(PROCESSED_DIR / 'transactions.parquet')
weekly = txn.groupby('week_no')['sales_value'].sum()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(weekly.index, weekly.values, color='#1f77b4', linewidth=1)
ax.set_title('Total Weekly Revenue — All Categories', fontsize=12)
ax.set_xlabel('Week Number', fontsize=10)
ax.set_ylabel('Revenue (USD)', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
out = CHARTS / 'weekly_revenue_all_categories.png'
fig.savefig(out, dpi=130)
plt.close(fig)
print(f'Chart saved to {out}')

## Summary — Data Cleaning

- **~2.6M transactions** across 2,500 panel households, spanning days 1–711 (~102 weeks / ~2 years).
- **7 tables** cleaned: products, transactions, demographics, campaign_desc, campaign_table, coupon, coupon_redempt.
- All column names lowercased; leading/trailing whitespace stripped from object columns.
- Demographics available for **801 of 2,500 households (32%)**; remaining 1,699 households are uncharacterized.
- Total store revenue: **$8,057,463** across all product categories.